In [1]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.utils import shuffle
from scipy.signal import resample
import matplotlib.pyplot as plt
import json
import warnings
import mne
from mne.preprocessing import ICA
warnings.filterwarnings("ignore")
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [2]:
SAMPLE_RATE = 200  # fs
# baseline correction according to paper, tmin and tmax was observed from the data
TMIN = -0.2 
TMAX = 0.8
BASELINE=(-0.2, 0.0)   
TASK_ID = 3  # Reinforcement Learning Task
sub_folder_path = str(SAMPLE_RATE) + 'Hz'
sub_folder_path

'200Hz'

In [3]:
## Load participants.tsv file
# root dir
root = "RLPD/"
participants_path = os.path.join(root, "participants.tsv")
participants = pd.read_csv(participants_path, sep="\t")
participants


,participant_id,Original_ID,Group,sess1_Med,sess2_Med,sex,age
0,sub-001,8010,CTL,NaN,no s2,Female,61
1,sub-002,801,PD,ON,OFF,Female,60
2,sub-003,802,PD,OFF,ON,Male,75
3,sub-004,803,PD,OFF,ON,Female,76
4,sub-005,804,PD,ON,OFF,Male,75
5,sub-006,805,PD,ON,OFF,Male,79
6,sub-007,8060,CTL,NaN,no s2,Female,83
7,sub-008,806,PD,OFF,ON,Female,79
8,sub-009,8070,CTL,NaN,no s2,Female,67
9,sub-010,807,PD,OFF,ON,Female,72


In [4]:
# Test for bad channels, sampling freq and shape
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(root):
    if sub.startswith("sub-"):   # Confirm it is a subject directory
        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):
            ses_path = os.path.join(sub_path, ses, "eeg")
            if os.path.exists(ses_path):
                for file in os.listdir(ses_path):
                    if file.endswith(".set"):
                        file_path = os.path.join(ses_path, file)
                        print(f"Reading in progress: {file_path}")
                        
                        try:
                            # Try loading raw first
                            raw = mne.io.read_raw_eeglab(file_path, preload=True)
                            bad_channel_list.append(raw.info['bads'])
                            sampling_freq_list.append(raw.info['sfreq'])
                            data_shape_list.append(raw.get_data().shape)
                        except Exception as e1:
                            print(f"  -> If raw fails, try epochs: {e1}")
                            try:
                                epochs = mne.read_epochs_eeglab(file_path)
                                epochs.load_data()
                                bad_channel_list.append(epochs.info['bads'])
                                sampling_freq_list.append(epochs.info['sfreq'])
                                data_shape_list.append(epochs.get_data().shape)
                            except Exception as e2:
                                print(f"  -> Failed to load epochs as well: {e2}")

Reading in progress: RLPD/sub-001\ses-01\eeg\sub-001_ses-01_task-ReinforcementLearning_eeg.set
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\RLPD\RLPD\sub-001\ses-01\eeg\sub-001_ses-01_task-ReinforcementLearning_eeg.fdt
Reading 0 ... 704899  =      0.000 ...  1409.798 secs...
Reading in progress: RLPD/sub-002\ses-01\eeg\sub-002_ses-01_task-ReinforcementLearning_eeg.set
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\RLPD\RLPD\sub-002\ses-01\eeg\sub-002_ses-01_task-ReinforcementLearning_eeg.fdt
Reading 0 ... 960124  =      0.000 ...  1920.248 secs...
Reading in progress: RLPD/sub-002\ses-02\eeg\sub-002_ses-02_task-ReinforcementLearning_eeg.set
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\RLPD\RLPD\sub-002\ses-02\eeg\sub-002_ses-02_task-ReinforcementLearning_eeg.fdt
Reading 0 ... 312124  =      0.000 ...   624.248 secs...
Reading in progress: RLPD/sub-003\ses-01\eeg\sub-003_ses-01_task-ReinforcementLearning_eeg.set
Reading D:\PycharmPorjects\DataProcess

In [5]:
from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
(67, 704900)
Channel number counter: Counter({67: 84})
Sampling rate counter: Counter({500.0: 84})


In [6]:
common_channels = []
for sub in os.listdir(root):
    if sub.startswith("sub-"):   # Only process subject directories
        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):  # Iterate over ses-xx directories
            ses_path = os.path.join(sub_path, ses, "eeg")
            if os.path.exists(ses_path):
                for file in os.listdir(ses_path):
                    if file.endswith(".set"):
                        file_path = os.path.join(ses_path, file)
                        print(f"Reading in progress: {file_path}")
                        
                        # Try loading raw data first, fallback to epochs if raw fails
                        try:
                            raw = mne.io.read_raw_eeglab(file_path, preload=True)
                            current_channels = set(raw.info['ch_names'])
                        except Exception as e1:
                            print(f"  -> Raw read failed, fallback to epochs: {e1}")
                            try:
                                epochs = mne.read_epochs_eeglab(file_path)
                                epochs.load_data()
                                current_channels = set(epochs.info['ch_names'])
                            except Exception as e2:
                                print(f"  -> Failed to read epochs as well: {e2}")
                                continue  # Skip this file
                        
                        # Find common channels
                        if not common_channels:
                            common_channels = current_channels
                        else:
                            common_channels &= current_channels

common_channels = list(common_channels)
print(f"Common channels ({len(common_channels)}):", common_channels)

Reading in progress: RLPD/sub-001\ses-01\eeg\sub-001_ses-01_task-ReinforcementLearning_eeg.set
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\RLPD\RLPD\sub-001\ses-01\eeg\sub-001_ses-01_task-ReinforcementLearning_eeg.fdt
Reading 0 ... 704899  =      0.000 ...  1409.798 secs...
Reading in progress: RLPD/sub-002\ses-01\eeg\sub-002_ses-01_task-ReinforcementLearning_eeg.set
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\RLPD\RLPD\sub-002\ses-01\eeg\sub-002_ses-01_task-ReinforcementLearning_eeg.fdt
Reading 0 ... 960124  =      0.000 ...  1920.248 secs...
Reading in progress: RLPD/sub-002\ses-02\eeg\sub-002_ses-02_task-ReinforcementLearning_eeg.set
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\RLPD\RLPD\sub-002\ses-02\eeg\sub-002_ses-02_task-ReinforcementLearning_eeg.fdt
Reading 0 ... 312124  =      0.000 ...   624.248 secs...
Reading in progress: RLPD/sub-003\ses-01\eeg\sub-003_ses-01_task-ReinforcementLearning_eeg.set
Reading D:\PycharmPorjects\DataProcess

In [7]:
# remove unreliable electrodes FT9, FT10, TP9, TP10 and non-EEG channels X, Y, Z, VEOG
common_channels = [ch for ch in common_channels if ch not in ['FT9', 'FT10', 'TP9', 'TP10', 'X', 'Y', 'Z', 'VEOG']]
print(common_channels)
print("Common channels number: ", len(common_channels))

['C5', 'CP3', 'P1', 'POz', 'C3', 'AF8', 'FT7', 'FT8', 'CP2', 'T7', 'Pz', 'F4', 'F5', 'FC2', 'FCz', 'O1', 'Oz', 'P5', 'F8', 'AF7', 'F1', 'P4', 'PO8', 'F2', 'FC4', 'PO3', 'C2', 'O2', 'Fp1', 'FC1', 'F6', 'P8', 'AFz', 'AF3', 'T8', 'Cz', 'AF4', 'FC6', 'P6', 'P7', 'C4', 'PO7', 'Fz', 'F3', 'C6', 'Fp2', 'C1', 'CP4', 'F7', 'FC3', 'PO4', 'TP7', 'CP6', 'CP5', 'TP8', 'CP1', 'P3', 'P2', 'FC5']
Common channels number:  59


In [8]:
def data_preprocessing(
        raw,
        common_channels: list,
        sample_rate: int = 250,
        notch_freq: float = 60.0,
        l_freq: float = 0.1,
        h_freq: float = 20.0,
        do_bad_interp: bool = True,
        verbose: bool = True,
):
    """
    Simon Conflict Task EEG Preprocessing:
      1) Select common channels, remove unreliable ones (FT9, FT10, TP9, TP10), reorder
      2) Set Montage
      3) 60 Hz Notch filter
      4) Bandpass filter (0.1–20 Hz)
      5) Interpolate bad channels (if enabled)
      6) Re-reference to average
      7) ICA removal (fit on 1 Hz copy + ICLabel)
      8) Downsample to 250 Hz (if needed)
    """

    # 1. Remove unreliable electrodes
    keep = common_channels
    raw.pick_channels(keep)
    raw.reorder_channels(keep)
    if verbose:
        print(f"✔ Step 1: Picked & reordered channels {keep}")

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'))
    if verbose:
        print("✔ Step 2: Montage set to 'standard_1020'.")

    # 3. Notch filter (60 Hz)
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch filtered at {notch_freq} Hz")

    # 4. Bandpass filter (0.1–20 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass filtered from {l_freq}–{h_freq} Hz")

    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bad channels: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bad channels to interpolate.")

    # 6. Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: EEG re-referenced to average")

    # 7. ICA
    raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, picks="eeg", verbose=False)
    ica = ICA(n_components=None, method="fastica", random_state=97, max_iter="auto")

    ica.fit(raw_for_ica)

    excluded = []
    if label_components is not None:
        try:
            ic_labels = label_components(raw_for_ica, ica, method="iclabel")
            labels = ic_labels["labels"]
            probs = ic_labels["y_pred_proba"]
            thresholds = {
                "eye blink": 0.7,
                "muscle artifact": 0.6,
                "heart beat": 0.5,
                "line noise": 0.8,
                "channel noise": 0.9,
            }
            for i, lab in enumerate(labels):
                if lab in thresholds:
                    p = probs[i].max() if probs is not None else 1.0
                    if p >= thresholds[lab]:
                        excluded.append(i)
        except Exception as e:
            if verbose:
                print(f"⚠ ICLabel failed: {e}")
    else:
        if verbose:
            print("ℹ ICLabel not available; ICA fitted but no auto exclusion")

    if excluded:
        ica.exclude = sorted(set(excluded))
        raw = ica.apply(raw.copy())
        if verbose:
            print(f"✔ Step 7: ICA applied. Excluded components: {ica.exclude}")
    else:
        if verbose:
            print("ℹ Step 7: No ICA components excluded.")

    # 8. Downsample (if not already 250 Hz)
    if raw.info["sfreq"] != sample_rate:
        raw.resample(sample_rate, npad="auto", verbose=False)
        if verbose:
            print(f"✔ Step 8: Resampled to {sample_rate} Hz")

    return raw

In [9]:
import numpy as np
import pandas as pd
import mne


def epoch_and_make_xy(
    raw: mne.io.Raw,
    events_tsv_path: str,
    tmin: float = -0.2,
    tmax: float = 0.8,
    baseline=(-0.2, 0),
    task_id: int = 1,
    subject_id: int = 1,
    disease_id: int = 0,
):
    """
    Epoch ds003506 training and test trials.

    Output:
        X: (n_trials, n_times, n_channels)
        y: (n_trials, 4) = [disease_id, stimulus_type, subject_id, task_id]

    stimulus_type:
        Training phase, feedback-locked:
            0 = choose_reward
            1 = choose_no_reward
            2 = match_reward
            3 = match_no_reward

        Test phase, response-locked:
            4 = test_left_response
            5 = test_right_response

    Event logic:
        Training:
            S 1  = Instr: "Choose"
            S 2  = Instr: "Match"
            S10  = FB: 0
            S11  = FB: +1

        Excluded training feedback:
            S 6 = FB: "No Match"
            S 7 = FB: "Too Slow"

        Test:
            S 4 = Left Button Choice
            S 5 = Right Button Choice
            These are treated as test responses only when they are not inside
            an active training trial.
    """

    ev = pd.read_csv(events_tsv_path, sep="\t").copy()
    ev = ev.reset_index(drop=True)

    required_cols = ["onset", "trial_type", "value"]
    for col in required_cols:
        if col not in ev.columns:
            raise ValueError(
                f"Column '{col}' not found in {events_tsv_path}. "
                f"Available columns: {list(ev.columns)}"
            )

    ev["trial_type_str"] = ev["trial_type"].astype(str).str.strip()
    ev["value_str"] = ev["value"].astype(str).str.strip()

    # Extract numeric marker from values such as "S  1", "S 10", "S 11".
    ev["value_num"] = (
        ev["value_str"]
        .str.extract(r"S\s*(\d+)", expand=False)
    )
    ev["value_num"] = pd.to_numeric(ev["value_num"], errors="coerce")

    rows = []

    current_training_condition = None
    inside_training_trial = False

    for _, row in ev.iterrows():
        v = row["value_num"]

        if pd.isna(v):
            continue

        v = int(v)
        onset = row["onset"]

        # --------------------------------------------------
        # Training instruction markers
        # --------------------------------------------------
        if v == 1:
            current_training_condition = "choose"
            inside_training_trial = True
            continue

        elif v == 2:
            current_training_condition = "match"
            inside_training_trial = True
            continue

        # --------------------------------------------------
        # Training feedback markers, feedback-locked
        # --------------------------------------------------
        elif v in [10, 11]:
            if not inside_training_trial or current_training_condition is None:
                continue

            if current_training_condition == "choose" and v == 11:
                stimulus_type = 0  # choose_reward
            elif current_training_condition == "choose" and v == 10:
                stimulus_type = 1  # choose_no_reward
            elif current_training_condition == "match" and v == 11:
                stimulus_type = 2  # match_reward
            elif current_training_condition == "match" and v == 10:
                stimulus_type = 3  # match_no_reward
            else:
                current_training_condition = None
                inside_training_trial = False
                continue

            rows.append({
                "onset": onset,
                "stimulus_type": stimulus_type,
                "phase": "training",
                "lock_type": "feedback",
                "condition": current_training_condition,
                "event_value": v,
                "trial_type": row["trial_type_str"],
            })

            current_training_condition = None
            inside_training_trial = False
            continue

        # --------------------------------------------------
        # Invalid training feedback markers
        # Exclude FB: "No Match" and FB: "Too Slow"
        # --------------------------------------------------
        elif v in [6, 7]:
            current_training_condition = None
            inside_training_trial = False
            continue

        # --------------------------------------------------
        # Button response markers
        # Training response events are skipped.
        # Test response events are response-locked.
        # --------------------------------------------------
        elif v in [4, 5]:
            if inside_training_trial:
                # This is a training response before feedback.
                # Do not epoch here.
                continue

            if v == 4:
                stimulus_type = 4  # test_left_response
            elif v == 5:
                stimulus_type = 5  # test_right_response
            else:
                continue

            rows.append({
                "onset": onset,
                "stimulus_type": stimulus_type,
                "phase": "test",
                "lock_type": "response",
                "condition": "test",
                "event_value": v,
                "trial_type": row["trial_type_str"],
            })

            continue

        else:
            continue

    s2 = pd.DataFrame(rows)

    if len(s2) == 0:
        print("Available trial_type values:")
        print(ev["trial_type"].value_counts(dropna=False).head(50))

        print("Available event values:")
        print(ev["value"].value_counts(dropna=False).head(50))

        raise ValueError(
            f"No valid training feedback or test response events found in {events_tsv_path}."
        )

    s2 = s2.dropna(subset=["onset", "stimulus_type"]).copy()
    s2["stimulus_type"] = s2["stimulus_type"].astype(int)

    sfreq = raw.info["sfreq"]
    print(f"Current sampling frequency: {sfreq} Hz")

    # Use onset seconds × current sampling rate.
    # Do not directly use sample column because raw may have been resampled.
    samples = np.round(
        s2["onset"].values.astype(float) * sfreq
    ).astype(int)

    events = np.c_[
        samples,
        np.zeros_like(samples, dtype=int),
        np.ones_like(samples, dtype=int),
    ]

    picks = mne.pick_types(
        raw.info,
        eeg=True,
        eog=False,
        exclude="bads",
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id={"event": 1},
        tmin=tmin,
        tmax=tmax,
        baseline=baseline,
        picks=picks,
        proj=False,
        preload=True,
        reject=None,
        verbose=False,
    )

    data = epochs.get_data()  # (N, C, T)
    target_len = int(round((tmax - tmin) * sfreq))

    if data.shape[-1] > target_len:
        data = data[..., :target_len]

    elif data.shape[-1] < target_len:
        pad = target_len - data.shape[-1]
        data = np.pad(
            data,
            ((0, 0), (0, 0), (0, pad)),
            mode="edge",
        )

    X = np.transpose(data, (0, 2, 1))  # (N, T, C)

    # Align labels if MNE drops out-of-bound epochs.
    stimulus_type = s2["stimulus_type"].values.astype(int)
    stimulus_type = stimulus_type[epochs.selection]

    y = np.column_stack([
        np.full_like(stimulus_type, disease_id),  # disease_id
        stimulus_type,                            # stimulus_type
        np.full_like(stimulus_type, subject_id),  # subject_id
        np.full_like(stimulus_type, task_id),     # task_id
    ])

    assert X.shape[0] == y.shape[0], (
        f"X/y mismatch: X={X.shape}, y={y.shape}"
    )

    label_names = {
        0: "choose_reward",
        1: "choose_no_reward",
        2: "match_reward",
        3: "match_no_reward",
        4: "test_left_response",
        5: "test_right_response",
    }

    print("Task: ds003506 training-feedback and test-response classification")
    print(f"Number of retained trials: {len(stimulus_type)}")
    print("Label distribution:")

    labels, counts = np.unique(stimulus_type, return_counts=True)
    for label, count in zip(labels, counts):
        print(f"  {label}: {label_names[label]} = {count}")

    print("Phase distribution:")
    phase_after_drop = s2["phase"].values[epochs.selection]
    print(pd.Series(phase_after_drop).value_counts())

    print("Lock type distribution:")
    lock_after_drop = s2["lock_type"].values[epochs.selection]
    print(pd.Series(lock_after_drop).value_counts())

    return X, y

## PASS 1: Find total number of trials across all subjects

In [10]:
N_total = 0  # total number of trials across all subjects/sessions, used to pre-allocate memmap

for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):
            if 'ses-' in ses:
                print(f"Current subject/session: {sub} / {ses}")
                eeg_path = os.path.join(sub_path, ses, 'eeg/')

                # Initialize file paths
                set_file_path, events_file_path = None, None

                # Traverse the eeg folder to find files
                for file in os.listdir(eeg_path):
                    if file.endswith('.set'):
                        set_file_path = os.path.join(eeg_path, file)
                    if 'events.tsv' in file:
                        events_file_path = os.path.join(eeg_path, file)

                # Error checking
                if set_file_path is None:
                    raise FileNotFoundError(f".set file not found in: {eeg_path}")
                if events_file_path is None:
                    raise FileNotFoundError(f"events.tsv not found in: {eeg_path}")

                # Load EEG data and preprocess
                print("Start preprocessing EEG data...")
                raw = mne.io.read_raw_eeglab(set_file_path, preload=True)
                # raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=60, l_freq=0.5, h_freq=40, verbose=True)


                # # Extract subject_id and disease_id

                subject_id = int(sub.split('-')[-1])
                disease_id = participants[participants['participant_id'] == sub]['Group'].values[0]
                if disease_id == 'CTL':
                    disease_id = 0
                elif disease_id == 'PD':
                    disease_id = 1

                print(f"Subject ID: {subject_id}, Disease ID: {disease_id}")
                print("Start epoching and making X, y...")

                # Remove beh_file_path, keep other parameters unchanged
                X, y = epoch_and_make_xy(
                    raw, events_file_path,
                    tmin=TMIN, tmax=TMAX, baseline=BASELINE,
                    task_id=TASK_ID, subject_id=subject_id, disease_id=disease_id
                )

                print(f"X shape: {X.shape}, y shape: {y.shape}")  
                # X: (N, SAMPLE_RATE * (tmax - tmin), C)
                # y: (N, 4) = [disease_id, stimulus_type, subject_id, task_id]

                N_total += X.shape[0]

                print("------------------------------------------------\n")

Current subject/session: sub-001 / ses-01
Start preprocessing EEG data...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\RLPD\RLPD\sub-001\ses-01\eeg\sub-001_ses-01_task-ReinforcementLearning_eeg.fdt
Reading 0 ... 704899  =      0.000 ...  1409.798 secs...
Subject ID: 1, Disease ID: 0
Start epoching and making X, y...
Current sampling frequency: 500.0 Hz
Task: ds003506 training-feedback and test-response classification
Number of retained trials: 252
Label distribution:
  0: choose_reward = 35
  1: choose_no_reward = 26
  2: match_reward = 36
  3: match_no_reward = 27
  4: test_left_response = 79
  5: test_right_response = 49
Phase distribution:
test        128
training    124
Name: count, dtype: int64
Lock type distribution:
response    128
feedback    124
Name: count, dtype: int64
X shape: (252, 500, 67), y shape: (252, 4)
------------------------------------------------

Current subject/session: sub-002 / ses-01
Start preprocessing EEG data...
Reading D:\PycharmPorjects\Da

In [11]:
output_root = os.path.join('Processed', sub_folder_path, 'RLPD')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
SEG_LEN = int((TMAX - TMIN) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 4))

X path: Processed\200Hz\RLPD\X.dat
y path: Processed\200Hz\RLPD\y.dat


## PASS 2: Process and store trials into memmap

In [12]:
cur = 0  # current index in memmap

for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub)
        for ses in os.listdir(sub_path):
            if 'ses-' in ses:
                print(f"Current subject/session: {sub} / {ses}")
                eeg_path = os.path.join(sub_path, ses, 'eeg/')

                # Initialize file paths
                set_file_path, events_file_path = None, None

                # Traverse the eeg folder to find files
                for file in os.listdir(eeg_path):
                    if file.endswith('.set'):
                        set_file_path = os.path.join(eeg_path, file)
                    if 'events.tsv' in file:
                        events_file_path = os.path.join(eeg_path, file)

                # Error checking
                if set_file_path is None:
                    raise FileNotFoundError(f".set file not found in: {eeg_path}")
                if events_file_path is None:
                    raise FileNotFoundError(f"events.tsv not found in: {eeg_path}")

                # Load EEG data and preprocess
                print("Start preprocessing EEG data...")
                raw = mne.io.read_raw_eeglab(set_file_path, preload=True)
                raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=60, l_freq=0.5, h_freq=40, verbose=True)


                # # Extract subject_id and disease_id

                subject_id = int(sub.split('-')[-1])
                disease_id = participants[participants['participant_id'] == sub]['Group'].values[0]
                if disease_id == 'CTL':
                    disease_id = 0
                elif disease_id == 'PD':
                    disease_id = 1

                print(f"Subject ID: {subject_id}, Disease ID: {disease_id}")
                print("Start epoching and making X, y...")

                # Remove beh_file_path, keep other parameters unchanged
                X, y = epoch_and_make_xy(
                    raw, events_file_path,
                    tmin=TMIN, tmax=TMAX, baseline=BASELINE,
                    task_id=TASK_ID, subject_id=subject_id, disease_id=disease_id
                )

                print(f"X shape: {X.shape}, y shape: {y.shape}")  
                # X: (N, SAMPLE_RATE * (tmax - tmin), C)
                # y: (N, 4) = [disease_id, stimulus_type, subject_id, task_id]

                # Save to memmap
                X_mm[cur:cur+X.shape[0]] = X
                y_mm[cur:cur+y.shape[0]] = y
                cur += X.shape[0]

                print("------------------------------------------------\n")
                
print("DONE: cur =", cur, " expected N_total =", N_total)

# flush
del X_mm
del y_mm  

Current subject/session: sub-001 / ses-01
Start preprocessing EEG data...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\RLPD\RLPD\sub-001\ses-01\eeg\sub-001_ses-01_task-ReinforcementLearning_eeg.fdt
Reading 0 ... 704899  =      0.000 ...  1409.798 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
✔ Step 1: Picked & reordered channels ['C5', 'CP3', 'P1', 'POz', 'C3', 'AF8', 'FT7', 'FT8', 'CP2', 'T7', 'Pz', 'F4', 'F5', 'FC2', 'FCz', 'O1', 'Oz', 'P5', 'F8', 'AF7', 'F1', 'P4', 'PO8', 'F2', 'FC4', 'PO3', 'C2', 'O2', 'Fp1', 'FC1', 'F6', 'P8', 'AFz', 'AF3', 'T8', 'Cz', 'AF4', 'FC6', 'P6', 'P7', 'C4', 'PO7', 'Fz', 'F3', 'C6', 'Fp2', 'C1', 'CP4', 'F7', 'FC3', 'PO4', 'TP7', 'CP6', 'CP5', 'TP8', 'CP1', 'P3', 'P2', 'FC5']
✔ Step 2: Montage set to 'standard_1020'.
✔ Step 3: Notch filtered at 60 Hz
✔ Step 4: Band-pass filtered from 0.5–40 Hz
ℹ Step 5: No bad channels to interpolate.
✔ Step 6: EEG re-referenced to average
Fitting ICA to data using 59 

## Create JSON file

In [13]:
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN,
    "TMAX": TMAX,
    "BASELINE": BASELINE,
    "MONTAGE": "standard_1020",
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {0: "CTL", 1: "PD"},
        "stimulus_type": {0: "choose_reward", 1: "choose_no_reward", 2: "match_reward", 3: "match_no_reward", 4: "test_left_response", 5: "test_right_response"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {3: "Reinforcement Learning Task"}
    }
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved meta: Processed\200Hz\RLPD\meta.json


## Load and check the processed data

In [14]:
# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 21510
T: 200
C: 59
SAMPLE_RATE: 200
TMIN: -0.2
TMAX: 0.8
BASELINE: [-0.2, 0.0]
MONTAGE: standard_1020
CHANNELS: ['C5', 'CP3', 'P1', 'POz', 'C3', 'AF8', 'FT7', 'FT8', 'CP2', 'T7', 'Pz', 'F4', 'F5', 'FC2', 'FCz', 'O1', 'Oz', 'P5', 'F8', 'AF7', 'F1', 'P4', 'PO8', 'F2', 'FC4', 'PO3', 'C2', 'O2', 'Fp1', 'FC1', 'F6', 'P8', 'AFz', 'AF3', 'T8', 'Cz', 'AF4', 'FC6', 'P6', 'P7', 'C4', 'PO7', 'Fz', 'F3', 'C6', 'Fp2', 'C1', 'CP4', 'F7', 'FC3', 'PO4', 'TP7', 'CP6', 'CP5', 'TP8', 'CP1', 'P3', 'P2', 'FC5']
LABELS: {'disease_id': {'0': 'CTL', '1': 'PD'}, 'stimulus_type': {'0': 'choose_reward', '1': 'choose_no_reward', '2': 'match_reward', '3': 'match_no_reward', '4': 'test_left_response', '5': 'test_right_response'}, 'subject_id': 'integer from 1 to number of subjects', 'task_id': {'3': 'Reinforcement Learning Task'}}
-------------------------------------
Subject ID 001: X shape=(252, 200, 59), y shape=(252, 4)
Subject ID 002: X shape=(416, 200, 59), y shape=(416, 4)
Subject ID 003: X shape=(5